#### Algoritmo

1. carregar base de dados original
2. preencher os valores vazios com a média dos valores
3. criar função para diplicar dados
    - filtrar por unidade gestora 
    - copiar todos os dados 
    - filtrar a data
    - extrair o mês
    - substituir o ano
    - concatenar
4. remover outliers extremos (>= 10000000)


In [257]:
import pandas as pd
import numpy as np

In [ ]:
# 1. Carregamento dos dados
data_path = '../data/data.csv'
# data = pd.read_csv('../data.csv', sep=',', encoding="utf-8")
df = pd.read_csv(data_path)
df['data'] = pd.to_datetime(df['data'])
df['origem_dado'] = 'real'

In [259]:
# 2. Cálculo das metas
# Média entre 2023 e 2025 para 2024
contagem_2023 = len(df[df['ano'] == 2023])
contagem_2025 = len(df[df['ano'] == 2025])
meta_2024 = int((contagem_2023 + contagem_2025) / 2)
atual_2024 = len(df[df['ano'] == 2024])
print(f'2023: {contagem_2023}; 2024: {atual_2024}, 2025: {contagem_2025}')
print(f'Meta 2024: {meta_2024}')

2023: 4241; 2024: 787, 2025: 2755
Meta 2024: 3498


In [260]:
contagem_2026 = len(df[df['ano'] == 2026])
print(f'2026: {contagem_2026}')

2026: 100


In [261]:
# Proporcionalidade 2026 (baseado na média mensal até abril)
# Nota: Ajustar conforme a necessidade de granularidade temporal
# dados_ate_abril = df[df['data'].dt.month <= 4]
# media_mensal_historica = len(dados_ate_abril) / 4
media_anual_historica = (contagem_2023 + meta_2024 + contagem_2025) / 3
media_mensal_historica = media_anual_historica / 12
meta_2026 = int(media_mensal_historica) * 4
print(f'Meta 2026: {meta_2026}')

Meta 2026: 1164


In [262]:
colunas = df.columns.to_list()
colunas_monetarias = ["valor_empenhado", "valor_liquidado", "valor_total"]
colunas_datas = ['data', 'ano']
cat_cols = list(set(colunas) - set(colunas_monetarias) - set(colunas_datas) - set(['origem_dado']))

#### Função 1

In [263]:
# def gerar_sinteticos(df_base, ano, quantidade):
#     if quantidade <= 0:
#         return pd.DataFrame()

#     # Identificar colunas categóricas automaticamente
#     cat_cols = [col for col in df_base.columns if col not in 
#                 ['valor_empenhado', 'valor_liquidado', 'valor_total', 'data', 'ano', 'origem_dado', 'n_empenho']]
    
#     lista_sinteticos = []
#     unidades_gestoras = df_base['unidade_gestora'].unique()

#     for _ in range(quantidade):
#         # Seleciona aleatoriamente uma unidade gestora para basear o dado
#         unid_gest = np.random.choice(unidades_gestoras)
#         df_filtro_ug = df_base[df_base['unidade_gestora'] == unid_gest]
        
#         linha = {}
#         for col in df_base.columns:
#             if col in ['valor_empenhado', 'valor_liquidado', 'valor_total']:
#                 # Gera valor baseado na média e std daquela UG, garantindo não ser negativo
#                 valor = np.random.normal(df_filtro_ug[col].quantile(0.10), df_filtro_ug[col].std())
#                 linha[col] = max(0, valor)
#             elif col == 'data':
#                 linha[col] = pd.Timestamp(year=ano, month=np.random.randint(1, 13), day=np.random.randint(1, 28))
#             elif col == 'ano':
#                 linha[col] = ano
#             elif col == 'origem_dado':
#                 linha[col] = 'sintetico'
#             elif col == 'n_empenho':
#                 linha[col] = f"{ano}SINT{np.random.randint(1000, 9999)}"
#             else:
#                 # Pega um valor aleatório existente na base para aquela coluna
#                 linha[col] = np.random.choice(df_base[col].dropna().unique())
        
#         lista_sinteticos.append(linha)

#     return pd.DataFrame(lista_sinteticos)

# # Exemplo de uso:
# # novos_dados = gerar_sinteticos(df, 2024, 100)

In [264]:
# # 4. Geração e concatenação
# # novos_2024 = gerar_sinteticos(df, 2024, max(0, meta_2024 - atual_2024))
# # novos_2026 = gerar_sinteticos(df, 2026, meta_2026 - len(df[df['ano'] == 2026]))

# novos_2024 = gerar_sinteticos(df, 2024, 3500)
# novos_2026 = gerar_sinteticos(df, 2026, 900)

# df_final = pd.concat([df, novos_2024, novos_2026], ignore_index=True)

#### Função 2

In [265]:
# TODO: adicionar geração de novos números de empenho

def gerar_sinteticos(df_base, ano):
    lista_df_sinteticos = []
    unidades_gestoras = df_base['unidade_gestora'].unique()

    for unid_gest in unidades_gestoras:
        df_ug = df_base[df_base['unidade_gestora'] == unid_gest]
        
        sample_size = max(1, int(0.1 * len(df_ug)))
        
        df_filtrado = df_ug.sample(sample_size).copy()
        
        if ano == 2026:
            meses = np.random.randint(1, 5, size=len(df_filtrado))
        else:
            meses = np.random.randint(1, 13, size=len(df_filtrado))
            
        df_filtrado['ano'] = ano
        dias = np.random.randint(1, 29, size=len(df_filtrado))
        
        df_filtrado['data'] = pd.to_datetime({
            'year': df_filtrado['ano'],
            'month': meses,
            'day': dias
        })
        
        df_filtrado['origem_dado'] = 'sintetico'
        lista_df_sinteticos.append(df_filtrado)
    
    if lista_df_sinteticos:
        df_final = pd.concat(lista_df_sinteticos, ignore_index=True)
        
        if ano == 2026 and len(df_final) > 1000:
            df_final = df_final.sample(1000, random_state=42)
            
        return df_final
    else:
        return pd.DataFrame()

In [266]:
novos_2024 = gerar_sinteticos(df, 2024)
novos_2025 = gerar_sinteticos(df, 2025)
novos_2026 = gerar_sinteticos(df, 2026)

df_final = pd.concat([df, novos_2024, novos_2025, novos_2026], ignore_index=True)

In [267]:
# # novos_2024 = gerar_sinteticos(df, 2024)
# novos_2025_2 = gerar_sinteticos(df, 2025)
# # novos_2026 = gerar_sinteticos(df, 2026)

# df_final = pd.concat([df_final, novos_2025_2], ignore_index=True)

In [268]:
# df_final.info()

In [269]:
df_final = df_final[df_final['data'] <= '2026-04-30']

In [270]:
df_final['ano'].value_counts()

ano
2023    4241
2025    3541
2024    1573
2026     883
Name: count, dtype: int64

In [271]:
df_final.to_csv('../data_expanded.csv', index=False)

In [272]:
# df_final[df_final['ano'] == 2023][colunas_monetarias].sort_values(by='valor_total')

In [273]:
# df_final[df_final['ano'] == 2024][colunas_monetarias].sort_values(by='valor_total')

In [274]:
# df_final[df_final['ano'] == 2025][colunas_monetarias].sort_values(by='valor_total', ascending=False)

In [275]:
df_final = df_final[df_final['valor_total'] <= 1000000]

In [278]:
df_final.info()

<class 'pandas.core.frame.DataFrame'>
Index: 10147 entries, 0 to 10240
Data columns (total 24 columns):
 #   Column                Non-Null Count  Dtype         
---  ------                --------------  -----         
 0   n_empenho             10147 non-null  object        
 1   unidade_gestora       10147 non-null  object        
 2   credor                10147 non-null  object        
 3   valor_empenhado       10147 non-null  float64       
 4   valor_liquidado       10147 non-null  float64       
 5   valor_total           10147 non-null  float64       
 6   funcao                10147 non-null  object        
 7   programa              10147 non-null  object        
 8   acao                  10147 non-null  object        
 9   subacao               10147 non-null  object        
 10  modalidade_empenho    10147 non-null  object        
 11  modalidade_licitacao  8718 non-null   object        
 12  n_licitacao           6535 non-null   object        
 13  obs_empenho          

In [279]:
df_final.to_csv('../data_expanded_3.csv', index=False)